In [1]:
# NOTEBOOK 23 - cross-city data audit (which Indian cities have enough usable PM2.5 stations?)
# Source: OpenAQ. Station list + coordinates come from the OpenAQ API (needs a free key in the environment
# variable OPENAQ_API_KEY); the hourly values come from the public OpenAQ S3 archive (no key needed).
# For each candidate city it counts reference-grade (monitor) stations with a PM2.5 sensor and measures how
# many hours of the chosen test windows actually have data. A station is "usable" if coverage >= 70 %.
# HOW TO CHECK: this cell only imports and sets options.
import os, json, io, gzip, time, datetime as dt
import numpy as np, pandas as pd, requests
from concurrent.futures import ThreadPoolExecutor

WORK = r'D:\FYP\Eco-Twin\v2_clean\crosscity_cache'
os.makedirs(WORK, exist_ok=True)
KEY = os.environ.get('OPENAQ_API_KEY', '')
API = 'https://api.openaq.org/v3'
S3  = 'https://openaq-data-archive.s3.amazonaws.com/records/csv.gz'

# windows that overlap the Delhi test period (8 Dec 2025 - 9 Apr 2026); two winter months keep the audit light
WINDOWS = [('2025-12-08', '2025-12-31'), ('2026-02-01', '2026-02-28')]
MIN_COVER = 0.70
HEADERS = {'X-API-Key': KEY}
print('API key set:', bool(KEY))

API key set: True


In [2]:
# candidate cities as latitude/longitude boxes (south, north, west, east)
CITIES = {
    'Mumbai':    (18.85, 19.35, 72.75, 73.15),
    'Kolkata':   (22.40, 22.75, 88.20, 88.50),
    'Hyderabad': (17.20, 17.65, 78.20, 78.65),
    'Bengaluru': (12.80, 13.15, 77.40, 77.80),
    'Chennai':   (12.85, 13.30, 80.10, 80.35),
    'Lucknow':   (26.70, 27.00, 80.80, 81.10),
    'Kanpur':    (26.35, 26.55, 80.20, 80.45),
    'Patna':     (25.50, 25.70, 85.00, 85.30),
    'Ahmedabad': (22.90, 23.15, 72.45, 72.70),
    'Jaipur':    (26.75, 27.05, 75.65, 75.95),
    'Pune':      (18.40, 18.65, 73.70, 74.00),
    'Gurugram/Noida/Faridabad (NCR)': (28.30, 28.65, 76.90, 77.60),
}
def which_city(lat, lon):
    for name, (s, n, w, e) in CITIES.items():
        if s <= lat <= n and w <= lon <= e:
            return name
    return None

# 1) every OpenAQ location in India (paged), cached on disk
cache = os.path.join(WORK, 'india_locations.json')
if os.path.exists(cache):
    locs = json.load(open(cache))
else:
    assert KEY, 'Set OPENAQ_API_KEY first (free key from explore.openaq.org), then restart the kernel.'
    locs, page = [], 1
    while True:
        r = requests.get(f'{API}/locations', headers=HEADERS, params={'countries_id': 9, 'limit': 1000, 'page': page}, timeout=60)
        r.raise_for_status()
        res = r.json()['results']
        locs += res
        if len(res) < 1000:
            break
        page += 1; time.sleep(1.2)
    json.dump(locs, open(cache, 'w'))
print('OpenAQ locations in India:', len(locs))

# HOW TO CHECK: prints several thousand locations (India has many low-cost sensors). A 401 means the key is missing or wrong.

OpenAQ locations in India: 775


In [3]:
# 2) keep fixed reference-grade monitors that have a PM2.5 sensor and sit inside a candidate city box
rows = []
for L in locs:
    c = L.get('coordinates') or {}
    lat, lon = c.get('latitude'), c.get('longitude')
    if lat is None or L.get('isMobile') or not L.get('isMonitor'):
        continue
    sens = [s for s in L.get('sensors', []) if (s.get('parameter') or {}).get('name') == 'pm25']
    city = which_city(lat, lon)
    if sens and city:
        rows.append({'location_id': L['id'], 'name': L['name'], 'city': city, 'lat': lat, 'lon': lon,
                     'provider': (L.get('provider') or {}).get('name'), 'first': (L.get('datetimeFirst') or {}).get('utc'),
                     'last': (L.get('datetimeLast') or {}).get('utc')})
cand = pd.DataFrame(rows)
print(cand.groupby('city').size().sort_values(ascending=False).to_string())
print('total candidate stations:', len(cand))

# HOW TO CHECK: Delhi is excluded on purpose. Cities with fewer than ~6 monitors here are unlikely to work.

city
Gurugram/Noida/Faridabad (NCR)    59
Mumbai                            44
Hyderabad                         17
Kolkata                           15
Bengaluru                         14
Pune                              14
Chennai                           13
Ahmedabad                         10
Lucknow                           10
Jaipur                             7
Kanpur                             7
Patna                              7
total candidate stations: 217


In [4]:
# 3) hourly coverage per station in the test windows, read from the public S3 archive
def day_urls(loc_id, start, end):
    d0, d1 = dt.date.fromisoformat(start), dt.date.fromisoformat(end)
    d = d0
    while d <= d1:
        yield f'{S3}/locationid={loc_id}/year={d.year}/month={d.month:02d}/location-{loc_id}-{d:%Y%m%d}.csv.gz'
        d += dt.timedelta(days=1)

def fetch_day(url):
    try:
        r = requests.get(url, timeout=40)
        if r.status_code != 200:
            return None
        df = pd.read_csv(io.BytesIO(r.content), compression='gzip')
        df = df[df['parameter'] == 'pm25']
        return df[['datetime', 'value']] if len(df) else None
    except Exception:
        return None

def coverage(loc_id):
    urls, total_hours = [], 0
    for a, b in WINDOWS:
        total_hours += int(((dt.datetime.fromisoformat(b) - dt.datetime.fromisoformat(a)).days + 1) * 24)
        urls += list(day_urls(loc_id, a, b))
    with ThreadPoolExecutor(max_workers=8) as ex2:
        parts = [p for p in ex2.map(fetch_day, urls) if p is not None]
    if not parts:
        return 0.0, 0.0
    d = pd.concat(parts)
    d['datetime'] = pd.to_datetime(d['datetime'], utc=True)
    d = d[(d['value'] >= 0) & (d['value'] < 1000)]                    # drop negatives and the 1000 ceiling
    hours = d['datetime'].dt.floor('h').nunique()
    return hours / total_hours, float(d['value'].median()) if len(d) else 0.0

cov_cache = os.path.join(WORK, 'coverage.json')
cov = json.load(open(cov_cache)) if os.path.exists(cov_cache) else {}
todo = [int(i) for i in cand['location_id'] if str(int(i)) not in cov]
print('stations to measure:', len(todo))
t0 = time.time()
with ThreadPoolExecutor(max_workers=6) as ex:
    for lid, res in zip(todo, ex.map(coverage, todo)):
        cov[str(lid)] = res
json.dump(cov, open(cov_cache, 'w'))
print(f'done in {time.time() - t0:.0f}s')

# HOW TO CHECK: takes a few minutes for ~100-200 stations (about 50 small files each). Re-running is instant (cached).

stations to measure: 217
done in 538s


In [5]:
# 4) per-city verdict
cand['coverage'] = cand['location_id'].map(lambda i: cov.get(str(int(i)), (0, 0))[0])
cand['median_pm25'] = cand['location_id'].map(lambda i: cov.get(str(int(i)), (0, 0))[1])
cand['usable'] = cand['coverage'] >= MIN_COVER
rep = (cand.groupby('city').agg(stations=('location_id', 'size'), usable=('usable', 'sum'),
                                median_cover=('coverage', 'median'), median_pm25=('median_pm25', 'median'))
       .sort_values('usable', ascending=False))
print(rep.round(2).to_string())
cand.sort_values(['city', 'coverage'], ascending=[True, False]).to_csv(os.path.join(WORK, 'crosscity_stations.csv'), index=False)
rep.to_csv(os.path.join(WORK, 'crosscity_city_report.csv'))

ok = rep[rep['usable'] >= 6]
print()
print('cities with at least 6 usable stations (minimum for a meaningful graph):', list(ok.index) or 'none')
print('best candidates: the top two rows above, preferring one city with winter pollution like Delhi (Kolkata, Lucknow, Kanpur, Patna)')
print('and one with a different climate (Mumbai, Chennai, Bengaluru, Hyderabad).')
print('saved crosscity_stations.csv and crosscity_city_report.csv in', WORK)

# HOW TO READ: "usable" = stations with >= 70 % of hours present in the test windows. Prefer cities with 8 or more.
# Also check median_pm25: very low values (< 40) mean the Delhi-trained scale will be far off and few-shot is needed.

                                stations  usable  median_cover  median_pm25
city                                                                       
Gurugram/Noida/Faridabad (NCR)        59      37          0.87        99.40
Mumbai                                44      27          0.82        44.67
Hyderabad                             17      12          0.84        35.83
Kolkata                               15      10          0.90        74.94
Ahmedabad                             10       9          0.90        56.62
Bengaluru                             14       8          0.78        34.88
Chennai                               13       7          0.85        31.04
Patna                                  7       7          0.86        61.20
Lucknow                               10       6          0.86        37.50
Jaipur                                 7       6          0.90        63.00
Pune                                  14       6          0.65        61.78
Kanpur      